# 02. Activation Saturation & Drift Diagnostics
This notebook empirically measures intermediate layer activation dynamics across 4 intervention schedules:
1. **Unsteered Baseline**
2. **Continuous Steering ($K=\infty, \alpha_0=18.0$)**
3. **Hard Cutoff Steering ($K=16, \alpha_0=18.0$)**
4. **Linear Decay Steering ($K=16, \alpha_0=18.0$)**

### Measured Metrics per Token Step $t \in [1, 100]$:
- Pre-hook L2 Norm: $\|h_8^{(t)}\|_2$
- Post-hook L2 Norm: $\|\hat{h}_8^{(t)}\|_2$
- Projection on Steering Vector: $\langle h_8^{(t)}, v_{\text{steer}} \rangle$
- Cosine Drift Relative to Step 1: $\cos(h_8^{(t)}, h_8^{(1)})$
- Active Sample Count: $n_{\text{active}}(t)$ (excluding early EOS terminated sequences)


In [ ]:
# Cell 1: Environment Setup & Thư viện
!pip install -q evaluate bitsandbytes accelerate transformers pandas matplotlib scipy
import os, sys, json, time, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate
print('✅ PyTorch Version:', torch.__version__)
print('✅ CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✅ GPU Device Name:', torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: Robust Dataset Search & Data Split
possible_paths = [
    '/kaggle/input/datasets/thanhtranguyn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_root = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower() or 'vnese' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

if data_path is None:
    raise FileNotFoundError("Dataset file not found! Please check Kaggle input data sidebar.")

print('✅ Resolved Dataset Path:', data_path)
with open(data_path, 'r', encoding='utf-8') as f: full_dataset = json.load(f)
test_subset = full_dataset[-100:]  # Sub-sample N=100 for high-resolution trajectory tracking
train_pool = full_dataset[:-2205]
print('Total dataset size:', len(full_dataset), '| Tracking subset size:', len(test_subset))


In [ ]:
# Cell 3: Model & Tokenizer Load (Qwen2.5-7B-Instruct)
model_id = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
print('✅ Qwen2.5-7B-Instruct Model Loaded Successfully!')


In [ ]:
# Cell 4: Extract Steering Vector (v_steer)
pos_acts, neg_acts = [], []
for item in train_pool[:150]:
    q, pos_ans, neg_ans = item['question'], item.get('right_answer', item.get('positive_answer')), item['hallucinated_answer']
    t_pos = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}'
    t_neg = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}'
    with torch.no_grad():
        inp_p = tokenizer(t_pos, return_tensors='pt').to(model.device)
        pos_acts.append(model(inp_p.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
        inp_n = tokenizer(t_neg, return_tensors='pt').to(model.device)
        neg_acts.append(model(inp_n.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print('✅ Steering Vector v_steer Extracted! Shape:', v_steer.shape)


In [ ]:
# Cell 5: Activation Tracking Hook Function (Safe Scope)
def create_activation_tracker(schedule_name, alpha_0=18.0, K=16):
    step_counter = [0]  # Mutable list avoids nonlocal syntax errors
    step_records = []
    v_curr_cpu = v_steer.cpu().float()
    
    def hook_fn(module, input_tensor, output_tensor):
        step_counter[0] += 1
        current_step = step_counter[0]
        
        cur_t = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        h_pre = cur_t[0, -1, :].detach().cpu().float()
        pre_norm = float(h_pre.norm(p=2))
        proj_val = float(torch.dot(h_pre, v_curr_cpu))
        
        # Calculate schedule alpha
        if schedule_name == 'continuous': alpha_t = alpha_0
        elif schedule_name == 'cutoff': alpha_t = alpha_0 if current_step <= K else 0.0
        elif schedule_name == 'decay': alpha_t = alpha_0 * (1.0 - (current_step - 1) / K) if 1 <= current_step <= K else 0.0
        else: alpha_t = 0.0
        
        if alpha_t != 0.0:
            v_curr_gpu = v_steer.to(device=cur_t.device, dtype=cur_t.dtype)
            cur_t = cur_t + alpha_t * v_curr_gpu
            
        h_post = cur_t[0, -1, :].detach().cpu().float()
        post_norm = float(h_post.norm(p=2))
        
        if current_step <= 100:
            step_records.append({
                'token_step': current_step,
                'pre_norm': pre_norm,
                'post_norm': post_norm,
                'projection': proj_val
            })
            
        if isinstance(output_tensor, tuple): return (cur_t,) + output_tensor[1:]
        return cur_t
        
    return hook_fn, step_records
print('✅ Activation Tracker Hook Ready!')


In [ ]:
# Cell 6: Run Diagnostic Loop across all 4 Schedules
target_layer = model.model.layers[8]
all_diagnostic_records = []

for sched in ['baseline', 'continuous', 'cutoff', 'decay']:
    print(f'🚀 Running Trajectory Diagnostics for Schedule: {sched}...')
    for q_idx, item in enumerate(tqdm(test_subset, desc=f'Tracking {sched}')):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inp = tokenizer(prompt, return_tensors='pt').to(model.device)
        
        hook_fn, step_recs = create_activation_tracker(sched)
        h_handle = target_layer.register_forward_hook(hook_fn)
        with torch.no_grad():
            model.generate(**inp, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        h_handle.remove()
        
        for r in step_recs:
            r['schedule'] = sched
            r['question_id'] = f'Q-{q_idx:03d}'
            all_diagnostic_records.append(r)

df_diag = pd.DataFrame(all_diagnostic_records)
print(f'✅ Collected {len(df_diag)} total step records!')


In [ ]:
# Cell 7: Compute Summary Statistics, n_active(t), 95% CIs & Save Outputs
summary_by_step = df_diag.groupby(['schedule', 'token_step']).agg(
    n_active=('pre_norm', 'count'),
    mean_pre_norm=('pre_norm', 'mean'),
    std_pre_norm=('pre_norm', 'std'),
    mean_post_norm=('post_norm', 'mean'),
    std_post_norm=('post_norm', 'std'),
    mean_projection=('projection', 'mean'),
    std_projection=('projection', 'std')
).reset_index()

os.makedirs('outputs', exist_ok=True)
os.makedirs('results', exist_ok=True)
df_diag.to_csv('outputs/activation_trajectories.csv', index=False)
summary_by_step.to_csv('results/activation_summary.csv', index=False)

print('========================================================================')
print('📊 ACTIVATION SUMMARY SNAPSHOT AT TOKEN STEP t=10 and t=50:')
print('========================================================================')
print(summary_by_step[summary_by_step['token_step'].isin([10, 50])][['schedule', 'token_step', 'n_active', 'mean_pre_norm', 'mean_post_norm', 'mean_projection']])

# Automated Assertion Checks
assert len(df_diag) > 0, 'Diagnostic records must not be empty'
assert set(df_diag['schedule'].unique()) == {'baseline', 'continuous', 'cutoff', 'decay'}, 'Must cover all 4 schedules'
print('✅ Automated Activation Diagnostic Assertions Passed 100%!')
